# Phase 8: measure trained inference with natural cleanup accounted for

This revised measurement uses the **same original notebook 05 `training.pt`** and all
8,192 validation queries. It compares reference, HGAM and blocks at batch sizes
**32, 128 and 512**, with five warmups and 20 paired repetitions, keeping the primary
comparison at size 32.

Notebook 07 showed that full Python garbage collection can add roughly a quarter-second
to an online pass. This protocol removes an avoidable reference cycle in the fingerprint
check and keeps **normal automatic collection enabled**. Every natural pause stays in
its actual online time. Minimal collection callbacks record cleanup durations; there is
no forced collection between passes and no per-stage host tracing.

Cleanup durations overlap measured latency and collection wall time: **do not add them
again or subtract them to manufacture a corrected speedup**. Experiment collection wall
includes warmups, audits, stage replays and reporting; it is not serving throughput.
Two independent V2 sessions passed 240 T4 tests and all 336 query batches each on
different physical T4 GPUs under the same contract. Primary latency reductions versus
HGAM were 8.38% and 8.59%, with separate within-run intervals 7.75–8.90% and 8.06–9.10%.
Size 512 is noisier in the repeat, including non-GC tails; all observations remain included.
Full collections still occur. The bounded comparison is complete, with no third unchanged
run planned. This notebook remains available for deliberate reproduction.
See [the verified two-run findings](https://github.com/WeaversMask/temporal-relational-gnn/blob/main/reports/GC_AWARE_INFERENCE_REPRODUCTION.md).

1. Start a fresh **T4 GPU** runtime and disconnect other managed runtimes.
2. Enter current budget readings below. Keep `RESUME = False` for a fresh reproduction.
3. Run cells in order. Upload the **same `training.pt` used for notebooks 06 and 07**.
4. After success or failure, record the ending balance, download the complete ZIP,
   and disconnect/delete the runtime. Keep every raw batch result.

Source reconstruction, full saved-validation replay and archived sampler fingerprints
must pass. Old benchmark code, checkpoints and notebook 06/07 results remain unchanged.
The setup/data replay and full paired study can take roughly as long as notebook 06;
duration varies. Independent reproduction is complete; any broader workload needs its own protocol.


In [ ]:
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
# Set True after checking that other managed runtimes are disconnected.
ISOLATED_RUNTIME_CONFIRMED = False
RESUME = False
CHECKPOINT_INPUT = None  # Own notebook 05 training.pt path, or None to upload it below.
RESUME_ARTIFACTS_DIR = None  # Only this revised protocol: extracted notebook 08 ZIP/batches path.
RESUME_RUN_ID = None  # On a fresh VM, optionally retain the ID from your earlier exported ZIP.
RESERVE_UNITS = 50
PROJECT_ROOT = Path("/content/temporal-relational-gnn")
DATA_DIR = PROJECT_ROOT / "data/rel-stack"
PROJECT_REF = "main"  # Same-folder reruns retain the first successful checkout for this protocol.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

# Preserve the same run folder if this cell is rerun in this notebook session.
if "_GC_INFERENCE_RUN_DIR" not in globals():
    if RESUME_RUN_ID is not None:
        if not RESUME or not isinstance(RESUME_RUN_ID, str) or not RESUME_RUN_ID:
            raise ValueError("RESUME_RUN_ID needs RESUME=True and the earlier run ID")
        if any(character not in "0123456789TZ-abcdef" for character in RESUME_RUN_ID):
            raise ValueError("Use only the run ID from the earlier exported ZIP filename")
    _GC_INFERENCE_RUN_ID = RESUME_RUN_ID or (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    )
    _GC_INFERENCE_RUN_DIR = (
        Path("/content/temporary-trgnn-gc-aware-inference/runs") / _GC_INFERENCE_RUN_ID
    )
    _GC_INFERENCE_RUN_DIR.mkdir(parents=True, exist_ok=False)
run_id = _GC_INFERENCE_RUN_ID
RUN_DIR = _GC_INFERENCE_RUN_DIR
CHECKPOINT_PATH = RUN_DIR / "checkpoints/training.pt"
TRAINING_REPORT = PROJECT_ROOT / "reports/colab/20261024T073010Z-1f9800/training-run.json"
print(f"Run directory: {RUN_DIR}", flush=True)
print(f"Checkpoint: {CHECKPOINT_PATH}", flush=True)


def run_logged(arguments, log_name, *, cwd=None, env=None):
    with (RUN_DIR / log_name).open("a") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT if cwd is None else cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)


def observed_nonnegative_number(name, value):
    if value is not None and (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or value < 0
    ):
        raise ValueError(f"{name} must be an observed non-negative number or None")


observed_nonnegative_number("COMPUTE_UNITS_PER_HOUR", COMPUTE_UNITS_PER_HOUR)
observed_nonnegative_number("COMPUTE_UNITS_REMAINING", COMPUTE_UNITS_REMAINING)
budget_path = RUN_DIR / "colab-budget.json"
if not budget_path.exists():
    budget_path.write_text(
        json.dumps(
            {
                "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
                "units_remaining_observed": COMPUTE_UNITS_REMAINING,
                "isolated_runtime_confirmed": ISOLATED_RUNTIME_CONFIRMED,
                "planned_total_units": 200,
                "reserve_units": RESERVE_UNITS,
                "scope": "current_runtime_interval_only",
                "resume_requested": RESUME,
            },
            indent=2,
            allow_nan=False,
        )
        + "\n"
    )
else:
    print("Preserving the original starting budget observation for this run folder.")
metadata_path = RUN_DIR / "notebook-run.json"
if metadata_path.exists():
    run_metadata = json.loads(metadata_path.read_text())
    if run_metadata["device"] != DEVICE:
        raise ValueError("Keep the same device for this profiling run folder")
else:
    run_metadata = {
        "notebook": "08_gc_aware_inference",
        "run_id": run_id,
        "device": DEVICE,
        "project_ref": PROJECT_REF,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
# Pin reruns of this run folder to the commit used by the first successful checkout.
EFFECTIVE_PROJECT_REF = run_metadata.get("source_commit", PROJECT_REF)
print("Project revision for this run:", EFFECTIVE_PROJECT_REF)
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError('Set DEVICE to "cpu" or "cuda"')
if COMPUTE_UNITS_REMAINING is not None and COMPUTE_UNITS_REMAINING <= RESERVE_UNITS:
    raise RuntimeError("The 50-unit reserve has been reached. Export the files and disconnect.")
if COMPUTE_UNITS_PER_HOUR is None or COMPUTE_UNITS_REMAINING is None:
    print("Unavailable budget observations are recorded as null.")

Clone the project and install the matching source dependencies. Reruns retain this run folder’s original source revision.


In [ ]:
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            'print("x-access-token" if "username" in sys.argv[1].lower() '
            'else os.environ["TRGNN_CLONE_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        clone_environment = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "TRGNN_CLONE_TOKEN": token,
        }
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        run_logged(command, "clone.log", cwd=Path("/content"), env=clone_environment)
except Exception as exc:
    (RUN_DIR / "setup-failure.json").write_text(
        json.dumps({"stage": "clone", "error_type": type(exc).__name__}, indent=2) + "\n"
    )
    raise
finally:
    token = None
    if "clone_environment" in globals():
        clone_environment.pop("TRGNN_CLONE_TOKEN", None)

run_logged(["git", "checkout", EFFECTIVE_PROJECT_REF], "checkout.log")
if EFFECTIVE_PROJECT_REF == "main":
    run_logged(["git", "merge", "--ff-only", "origin/main"], "checkout.log")
run_logged(["git", "rev-parse", "HEAD"], "checkout.log")
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if run_metadata.get("source_commit", source_commit) != source_commit:
    raise RuntimeError("This run folder must retain its original source commit")
run_metadata["source_commit"] = source_commit
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
run_logged(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    "install.log",
)

Check the actual sampler, GPU and complete correctness suite before loading the saved model.


In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

Freeze the revised protocol and verify the original checkpoint report and benchmark source. The model and sampling settings remain unchanged.


In [ ]:
import yaml

INFERENCE_CONFIG = json.loads((PROJECT_ROOT / "configs/gc_aware_inference.json").read_text())
frozen_config_path = RUN_DIR / "frozen-gc-aware-config.json"
if frozen_config_path.exists():
    if json.loads(frozen_config_path.read_text()) != INFERENCE_CONFIG:
        raise RuntimeError("This run folder must retain its original frozen configuration")
else:
    frozen_config_path.write_text(json.dumps(INFERENCE_CONFIG, indent=2) + "\n")
print(json.dumps(INFERENCE_CONFIG, indent=2))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if sha256_file(TRAINING_REPORT) != INFERENCE_CONFIG["training_report_sha256"]:
    raise RuntimeError("The parent training report must match the frozen report SHA256")
parent_report = json.loads(TRAINING_REPORT.read_text())
if parent_report.get("status") != "passed" or not parent_report.get("trained"):
    raise RuntimeError("The parent must be the successful trained notebook 05 run")
if parent_report["checkpoint"]["sha256"] != INFERENCE_CONFIG["checkpoint_sha256"]:
    raise RuntimeError("The parent checkpoint hash differs from the frozen configuration")

SOURCE_PROFILE = PROJECT_ROOT / INFERENCE_CONFIG["source_profile"]
if sha256_file(SOURCE_PROFILE) != INFERENCE_CONFIG["source_profile_sha256"]:
    raise RuntimeError("The reviewed source profile changed")
source_profile = json.loads(SOURCE_PROFILE.read_text())
if (
    source_profile["benchmark_contract_sha256"]
    != INFERENCE_CONFIG["source_benchmark_contract_sha256"]
):
    raise RuntimeError("The reviewed source benchmark contract changed")

pins = json.loads((PROJECT_ROOT / "configs/upstream.json").read_text())
source_receipt = DATA_DIR / ".trgnn-source.json"
try:
    cached_source = json.loads(source_receipt.read_text())
except (OSError, ValueError):
    cached_source = None
required_inputs = [
    DATA_DIR / "manifest.yaml",
    DATA_DIR / "tasks/user-badge/manifest.yaml",
    DATA_DIR / "tasks/user-badge/train.parquet",
    DATA_DIR / "tasks/user-badge/val.parquet",
]
if (DATA_DIR / "manifest.yaml").is_file():
    database_spec = yaml.safe_load((DATA_DIR / "manifest.yaml").read_text())
    required_inputs.extend(DATA_DIR / "db" / f"{name}.parquet" for name in database_spec["tables"])
NEEDS_DOWNLOAD = cached_source != pins["dataset"] or any(not p.is_file() for p in required_inputs)
print("Pinned training/validation inputs need download:", NEEDS_DOWNLOAD)

Upload the same original notebook 05 `training.pt` as before. The hash check rejects an incorrect checkpoint.


In [ ]:
checkpoint_candidate = CHECKPOINT_INPUT
if checkpoint_candidate is None and CHECKPOINT_PATH.is_file():
    checkpoint_candidate = CHECKPOINT_PATH
if checkpoint_candidate is None:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose the single training.pt checkpoint from notebook 05's ZIP")
    _, checkpoint_bytes = next(iter(uploaded.items()))
    restored = Path("/content/trgnn-gc-inference-import") / run_id / "training.pt"
    restored.parent.mkdir(parents=True, exist_ok=True)
    restored.write_bytes(checkpoint_bytes)
    checkpoint_candidate = restored
    del checkpoint_bytes, uploaded
checkpoint_candidate = Path(checkpoint_candidate)
if not checkpoint_candidate.is_file():
    raise FileNotFoundError(f"No notebook 05 checkpoint at {checkpoint_candidate}")
checkpoint_sha256 = sha256_file(checkpoint_candidate)
if checkpoint_sha256 != INFERENCE_CONFIG["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint SHA256 does not match the reviewed notebook 05 training.pt")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
if checkpoint_candidate.resolve() != CHECKPOINT_PATH.resolve():
    shutil.copy2(checkpoint_candidate, CHECKPOINT_PATH)
checkpoint_provenance = {
    "sha256": checkpoint_sha256,
    "bytes": CHECKPOINT_PATH.stat().st_size,
    "training_report_sha256": INFERENCE_CONFIG["training_report_sha256"],
    "parent_training_run": INFERENCE_CONFIG["parent_training_run"],
    "use": "fixed_model_gc_aware_inference_only",
}
(RUN_DIR / "checkpoint-input.json").write_text(json.dumps(checkpoint_provenance, indent=2) + "\n")
if RESUME_ARTIFACTS_DIR is not None:
    if not RESUME:
        raise ValueError("RESUME_ARTIFACTS_DIR requires RESUME=True")
    restored_batches = Path(RESUME_ARTIFACTS_DIR)
    if not restored_batches.is_dir():
        raise FileNotFoundError(f"No extracted profiling batches directory at {restored_batches}")
    batch_files = sorted(path for path in restored_batches.glob("batch-*.json") if path.is_file())
    if not batch_files:
        raise ValueError("The extracted profiling directory contains no batch-*.json artifacts")
    destination = RUN_DIR / "batches"
    destination.mkdir(exist_ok=True)
    for artifact in batch_files:
        target = destination / artifact.name
        if target.exists() and target.read_bytes() != artifact.read_bytes():
            raise RuntimeError(f"Conflicting restored batch artifact: {artifact.name}")
        if artifact.resolve() != target.resolve():
            shutil.copy2(artifact, target)
    print("Restored completed-batch candidates:", len(batch_files))
    print("The revised profiler rejects historical notebook 06 or diagnostic sidecars.")
print("Verified notebook 05 checkpoint:", CHECKPOINT_PATH)
print("SHA256:", checkpoint_sha256)

Run the new natural-GC measurement. It saves each completed batch atomically, retaining all pauses and separate GC accounting. The callbacks add observation overhead, recorded in this new protocol.


In [ ]:
arguments = [
    sys.executable,
    "-m",
    "trgnn.gc_profile",
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--training-report",
    str(TRAINING_REPORT),
    "--source-profile",
    str(SOURCE_PROFILE),
    "--data-dir",
    str(DATA_DIR),
    "--device",
    DEVICE,
    "--batch-sizes",
    *map(str, INFERENCE_CONFIG["batch_sizes"]),
    "--repeats",
    str(INFERENCE_CONFIG["repeats"]),
    "--warmup",
    str(INFERENCE_CONFIG["warmup"]),
    "--seed",
    str(INFERENCE_CONFIG["seed"]),
    "--bootstrap-samples",
    str(INFERENCE_CONFIG["bootstrap_samples"]),
    "--output",
    str(RUN_DIR / "gc-aware-profile.json"),
    "--artifacts-dir",
    str(RUN_DIR / "batches"),
]
if NEEDS_DOWNLOAD:
    arguments.append("--download")
if RESUME:
    arguments.append("--resume")
run_logged(arguments, "gc-aware-profile.log")

Inspect completion, equivalence checks and summaries for all three sizes. Retain every observation. Cleanup is already included wherever it overlaps a timer; GC durations are explanatory accounting, not an amount to add or subtract.


In [ ]:
report_path = RUN_DIR / "gc-aware-profile.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Status:", report.get("status"))
    print(
        "Completed query batches:",
        report.get("completed_batches", 0),
        "/",
        report.get("expected_batches", "?"),
    )
    replay = report.get("validation_replay", {})
    print("Saved prediction replay:", replay.get("status", "not completed"))
    print("Validation queries checked:", replay.get("validated_query_count", 0))
    print("Replay maximum prediction error:", replay.get("max_absolute_logit_error"))
    for size, summary in report.get("summary_by_batch_size", {}).items():
        print(f"\nBatch size {size}: {summary['query_batches']} query batches")
        for mode, values in summary["modes"].items():
            latency = values["batch_latency_ms"]
            throughput = values["throughput_queries_per_second"]
            print(
                f"  {mode}: median {latency['median']:.3f} ms, "
                f"p95 {latency['p95']:.3f} ms, {throughput:.1f} queries/s"
            )
        comparison = summary["comparisons"]["hgam"]
        reduction = comparison["blocks_latency_reduction_percent"]
        print(f"  Blocks latency reduction versus HGAM: {reduction:+.2f}% (positive is faster)")
        print("  95% interval:", comparison["cluster_bootstrap_95_percent_ci"])
        print("  Maximum prediction errors:", summary.get("max_absolute_logit_error", {}))
        print(
            "Cleanup accounting:",
            json.dumps(summary.get("gc_accounting", summary.get("gc_telemetry", {})), indent=2),
        )
    print(
        "Collection program wall:",
        json.dumps(report.get("collection_program_window", {}), indent=2),
    )
    print("Measurement protocol:", report.get("gc_version"))
    print(
        "Natural cleanup remains included in online latency. Do not add GC durations to it again."
    )
    print("GPU activity recording:", report.get("telemetry", {}).get("status", "not started"))
    if "error" in report:
        print("Error:", report["error"])
    print("Keep the complete ZIP: raw measurements and audits are saved alongside this summary.")
else:
    print("No profiler report yet; export the available setup logs and input checkpoint.")

Record the actual ending balance after the measurement finishes or stops.


In [ ]:
ENDING_UNITS_REMAINING = None
observed_nonnegative_number("ENDING_UNITS_REMAINING", ENDING_UNITS_REMAINING)
starting_budget = json.loads((RUN_DIR / "colab-budget.json").read_text())
starting_balance = starting_budget["units_remaining_observed"]
account_delta = (
    round(starting_balance - ENDING_UNITS_REMAINING, 2)
    if starting_balance is not None and ENDING_UNITS_REMAINING is not None
    else None
)
isolated_delta = (
    account_delta
    if starting_budget.get("isolated_runtime_confirmed")
    and account_delta is not None
    and account_delta >= 0
    else None
)
ending_budget = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "starting_observation_at_utc": starting_budget.get("recorded_at_utc"),
    "starting_units_remaining_observed": starting_balance,
    "ending_units_remaining_observed": ENDING_UNITS_REMAINING,
    "account_balance_decrease_units": account_delta,
    "balance_delta_precision_decimal_places": 2,
    "isolated_run_consumption_units": isolated_delta,
    "isolated_runtime_confirmed": starting_budget.get("isolated_runtime_confirmed", False),
    "scope": "current_runtime_interval_only",
    "note": (
        "Attribution needs observed balances and an isolated runtime; "
        "earlier intervals are separate."
    ),
}
(RUN_DIR / "budget-end.json").write_text(
    json.dumps(ending_budget, indent=2, allow_nan=False) + "\n"
)
print(json.dumps(ending_budget, indent=2))
if ENDING_UNITS_REMAINING is not None and ENDING_UNITS_REMAINING <= RESERVE_UNITS:
    print("The 50-unit reserve has been reached. Export and disconnect; do not start another run.")

Download the full run folder after success or failure, before disconnecting. It includes the input checkpoint, logs, frozen settings, summaries and raw batch/GC records.


In [ ]:
from google.colab import files

archive = shutil.make_archive(
    "/content/trgnn-gc-aware-inference-" + run_id, "zip", root_dir=str(RUN_DIR)
)
files.download(archive)